In [1]:
import os
import torch
import onnx
import onnxruntime
import numpy as np
import json
from onnx import numpy_helper,shape_inference
import onnx_graphsurgeon as gs
from polygraphy.backend.onnx.loader import fold_constants
import tempfile
import re
from utils_modelopt import (
    convert_zp_fp8,
    cast_resize_io,
    convert_fp16_io,
    cast_fp8_mha_io,
)
from onnxmltools.utils.float16_converter import convert_float_to_float16


/home/thanhdh/.conda/envs/idm-dev/lib/python3.10/site-packages/onnxscript/converter.py:820: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/home/thanhdh/.conda/envs/idm-dev/lib/python3.10/site-packages/onnxscript/converter.py:820: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()


In [2]:
class Optimizer():
    def __init__(
        self,
        onnx_graph,
        verbose=False
    ):
        self.graph = gs.import_onnx(onnx_graph)
        self.verbose = verbose

    def info(self, prefix):
        if self.verbose:
            print(f"{prefix} .. {len(self.graph.nodes)} nodes, {len(self.graph.tensors().keys())} tensors, {len(self.graph.inputs)} inputs, {len(self.graph.outputs)} outputs")

    def cleanup(self, return_onnx=False):
        self.graph.cleanup().toposort()
        return gs.export_onnx(self.graph) if return_onnx else self.graph

    def select_outputs(self, keep, names=None):
        self.graph.outputs = [self.graph.outputs[o] for o in keep]
        if names:
            for i, name in enumerate(names):
                self.graph.outputs[i].name = name

    def fold_constants(self, return_onnx=False):
        onnx_graph = fold_constants(gs.export_onnx(self.graph), allow_onnxruntime_shape_inference=True)
        self.graph = gs.import_onnx(onnx_graph)
        if return_onnx:
            return onnx_graph

    def infer_shapes(self, return_onnx=False):
        onnx_graph = gs.export_onnx(self.graph)
        if onnx_graph.ByteSize() > 2147483648:
            temp_dir = tempfile.TemporaryDirectory().name
            os.makedirs(temp_dir, exist_ok=True)
            onnx_orig_path = os.path.join(temp_dir, 'model.onnx')
            onnx_inferred_path = os.path.join(temp_dir, 'inferred.onnx')
            onnx.save_model(onnx_graph,
                onnx_orig_path,
                save_as_external_data=True,
                all_tensors_to_one_file=True,
                convert_attribute=False)
            onnx.shape_inference.infer_shapes_path(onnx_orig_path, onnx_inferred_path)
            onnx_graph = onnx.load(onnx_inferred_path)
        else:
            onnx_graph = shape_inference.infer_shapes(onnx_graph)

        self.graph = gs.import_onnx(onnx_graph)
        if return_onnx:
            return onnx_graph

    def clip_add_hidden_states(self, hidden_layer_offset, return_onnx=False):
        hidden_layers = -1
        onnx_graph = gs.export_onnx(self.graph)
        for i in range(len(onnx_graph.graph.node)):
            for j in range(len(onnx_graph.graph.node[i].output)):
                name = onnx_graph.graph.node[i].output[j]
                if "layers" in name:
                    hidden_layers = max(int(name.split(".")[1].split("/")[0]), hidden_layers)
        for i in range(len(onnx_graph.graph.node)):
            for j in range(len(onnx_graph.graph.node[i].output)):
                if onnx_graph.graph.node[i].output[j] == "/text_model/encoder/layers.{}/Add_1_output_0".format(hidden_layers+hidden_layer_offset):
                    onnx_graph.graph.node[i].output[j] = "hidden_states"
            for j in range(len(onnx_graph.graph.node[i].input)):
                if onnx_graph.graph.node[i].input[j] == "/text_model/encoder/layers.{}/Add_1_output_0".format(hidden_layers+hidden_layer_offset):
                    onnx_graph.graph.node[i].input[j] = "hidden_states"
        if return_onnx:
            return onnx_graph

    def fuse_mha_qkv_int8_sq(self):
        tensors = self.graph.tensors()
        keys = tensors.keys()

        # mha  : fuse QKV QDQ nodes
        # mhca : fuse KV QDQ nodes
        q_pat = (
            "/down_blocks.\\d+/attentions.\\d+/transformer_blocks"
            ".\\d+/attn\\d+/to_q/input_quantizer/DequantizeLinear_output_0"
        )
        k_pat = (
            "/down_blocks.\\d+/attentions.\\d+/transformer_blocks"
            ".\\d+/attn\\d+/to_k/input_quantizer/DequantizeLinear_output_0"
        )
        v_pat = (
            "/down_blocks.\\d+/attentions.\\d+/transformer_blocks"
            ".\\d+/attn\\d+/to_v/input_quantizer/DequantizeLinear_output_0"
        )

        qs = list(sorted(map(
            lambda x: x.group(0),  # type: ignore
            filter(lambda x: x is not None, [re.match(q_pat, key) for key in keys]),
            )))
        ks = list(sorted(map(
            lambda x: x.group(0),  # type: ignore
            filter(lambda x: x is not None, [re.match(k_pat, key) for key in keys]),
            )))
        vs = list(sorted(map(
            lambda x: x.group(0),  # type: ignore
            filter(lambda x: x is not None, [re.match(v_pat, key) for key in keys]),
            )))

        removed = 0
        assert len(qs) == len(ks) == len(vs), "Failed to collect tensors"
        for q, k, v in zip(qs, ks, vs):
            is_mha = all(["attn1" in tensor for tensor in [q, k, v]])
            is_mhca = all(["attn2" in tensor for tensor in [q, k, v]])
            assert (is_mha or is_mhca) and (not (is_mha and is_mhca))

            if is_mha:
                tensors[k].outputs[0].inputs[0] = tensors[q]
                tensors[v].outputs[0].inputs[0] = tensors[q]
                del tensors[k]
                del tensors[v]
                removed += 2
            else:  # is_mhca
                tensors[k].outputs[0].inputs[0] = tensors[v]
                del tensors[k]
                removed += 1
        print(f"Removed {removed} QDQ nodes")
        return removed # expected 72 for L2.5

    def modify_fp8_graph(self, is_fp16_io=True):
        onnx_graph = gs.export_onnx(self.graph)
        # Convert INT8 Zero to FP8.
        onnx_graph = convert_zp_fp8(onnx_graph)
        # Convert weights and activations to FP16 and insert Cast nodes in FP8 MHA.
        onnx_graph = convert_float_to_float16(onnx_graph, keep_io_types=True, disable_shape_infer=True)
        self.graph = gs.import_onnx(onnx_graph)
        # Add cast nodes to Resize I/O.
        cast_resize_io(self.graph)
        # Convert model inputs and outputs to fp16 I/O.
        if is_fp16_io:
            convert_fp16_io(self.graph)
        # Add cast nodes to MHA's BMM1 and BMM2's I/O.
        cast_fp8_mha_io(self.graph)


In [3]:
def optimize(onnx_graph,name, return_onnx=True, **kwargs):
    opt = Optimizer(onnx_graph, verbose=True)
    opt.info(name + ': original')
    opt.cleanup()
    opt.info(name + ': cleanup')
    if kwargs.get('modify_fp8_graph', False):
        is_fp16_io = kwargs.get('is_fp16_io', True)
        opt.modify_fp8_graph(is_fp16_io=is_fp16_io)
        opt.info(name + ': modify fp8 graph')
    opt.fold_constants()
    opt.info(name + ': fold constants')
    opt.infer_shapes()
    opt.info(name + ': shape inference')
    if kwargs.get('fuse_mha_qkv_int8', False):
        opt.fuse_mha_qkv_int8_sq()
        opt.info(name + ': fuse QKV nodes')
    onnx_opt_graph = opt.cleanup(return_onnx=return_onnx)
    opt.info(name + ': finished')
    return onnx_opt_graph

## Export Onnx and Optimize

In [5]:
onnx_path = {
    'model_name':"onnx_path"
}

In [4]:
def export_onnx(
        model,
        name,
        input_names,
        output_names,
        # dynamic_axes,
        sample_input,
        onnx_dir,
        onnx_opt_dir,
        onnx_opset,
        # static_shape=False,
    ):
        onnx_path = os.path.join(onnx_dir, 'model.onnx')
        onnx_opt_path = os.path.join(onnx_opt_dir, 'model.onnx')
        onnx_opt_graph = None
        # Export optimized ONNX model (if missing)
        if not os.path.exists(onnx_opt_path):
            os.mkdir(onnx_opt_dir)
            # Export ONNX model (if missing)
            if not os.path.exists(onnx_path):
                os.mkdir(onnx_dir)
                print(f"[I] Exporting ONNX model: {onnx_path}")
                def export_onnx(model):
                    inputs = sample_input
                    # torch.onnx.dynamo_export(model,
                    torch.onnx.export(model,
                        inputs,
                        onnx_path,
                        export_params=True,
                        opset_version=onnx_opset,
                        do_constant_folding=True,
                        input_names=input_names,
                        output_names=output_names,
                        dynamic_axes=None,
                    )
                with torch.inference_mode(), torch.autocast("cuda"):
                    export_onnx(model)
            else:
                print(f"[I] Found cached ONNX model: {onnx_path}")

            print(f"[I] Optimizing ONNX model: {onnx_opt_path}")
            onnx_opt_graph = optimize(onnx.load(onnx_path),name)
            if onnx_opt_graph.ByteSize() > 2147483648:
                onnx.save_model(
                    onnx_opt_graph,
                    onnx_opt_path,
                    save_as_external_data=True,
                    all_tensors_to_one_file=True,
                    convert_attribute=False)
            else:
                onnx.save(onnx_opt_graph, onnx_opt_path)
        else:
            print(f"[I] Found cached optimized ONNX model: {onnx_opt_path} ")

In [26]:
# Helper utility for weights map
def export_weights_map(self, onnx_opt_path, weights_map_path):
    if not os.path.exists(weights_map_path):
        onnx_opt_dir = os.path.dirname(onnx_opt_path)
        onnx_opt_model = onnx.load(onnx_opt_path)
        state_dict = self.get_model().state_dict()
        # Create initializer data hashes
        initializer_hash_mapping = {}
        for initializer in onnx_opt_model.graph.initializer:
            initializer_data = numpy_helper.to_array(initializer, base_dir=onnx_opt_dir).astype(np.float16)
            initializer_hash = hash(initializer_data.data.tobytes())
            initializer_hash_mapping[initializer.name] = (initializer_hash, initializer_data.shape)

        weights_name_mapping = {}
        weights_shape_mapping = {}
        # set to keep track of initializers already added to the name_mapping dict
        initializers_mapped = set()
        for wt_name, wt in state_dict.items():
            # get weight hash
            wt = wt.cpu().detach().numpy().astype(np.float16)
            wt_hash = hash(wt.data.tobytes())
            wt_t_hash = hash(np.transpose(wt).data.tobytes())

            for initializer_name, (initializer_hash, initializer_shape) in initializer_hash_mapping.items():
                # Due to constant folding, some weights are transposed during export
                # To account for the transpose op, we compare the initializer hash to the
                # hash for the weight and its transpose
                if wt_hash == initializer_hash or wt_t_hash == initializer_hash:
                    # The assert below ensures there is a 1:1 mapping between
                    # PyTorch and ONNX weight names. It can be removed in cases where 1:many
                    # mapping is found and name_mapping[wt_name] = list()
                    assert initializer_name not in initializers_mapped
                    weights_name_mapping[wt_name] = initializer_name
                    initializers_mapped.add(initializer_name)
                    is_transpose = False if wt_hash == initializer_hash else True
                    weights_shape_mapping[wt_name] = (initializer_shape, is_transpose)

            # Sanity check: Were any weights not matched
            if wt_name not in weights_name_mapping:
                print(f'[I] PyTorch weight {wt_name} not matched with any ONNX initializer')
        print(f'[I] {len(weights_name_mapping.keys())} PyTorch weights were matched with ONNX initializers')
        assert weights_name_mapping.keys() == weights_shape_mapping.keys()
        with open(weights_map_path, 'w') as fp:
            json.dump([weights_name_mapping, weights_shape_mapping], fp)
    else:
        print(f"[I] Found cached weights map: {weights_map_path} ")

### Convert Unet Encoder to Onnx

In [ ]:
import sys
sys.path.append('/workspace/Try-on-Product/projects/Leffa/')

In [6]:
pretrained_model_name_or_path='/workspace/Try-on-Product/projects/Leffa/ckpts/stable-diffusion-inpainting'

In [29]:
def replace_conv_in_layer(unet_model, new_in_channels):
    original_conv_in = unet_model.conv_in

    if original_conv_in.in_channels == new_in_channels:
        return original_conv_in

    new_conv_in = torch.nn.Conv2d(
        in_channels=new_in_channels,
        out_channels=original_conv_in.out_channels,
        kernel_size=original_conv_in.kernel_size,
        padding=1,
    )
    new_conv_in.weight.data.zero_()
    new_conv_in.bias.data = original_conv_in.bias.data.clone()
    if original_conv_in.in_channels < new_in_channels:
        new_conv_in.weight.data[:, : original_conv_in.in_channels] = (
            original_conv_in.weight.data
        )
    else:
        new_conv_in.weight.data[:, :new_in_channels] = original_conv_in.weight.data[
            :, :new_in_channels
        ]
    return new_conv_in

def replace_conv_out_layer(unet_model, new_out_channels):
    original_conv_out = unet_model.conv_out

    if original_conv_out.out_channels == new_out_channels:
        return original_conv_out

    new_conv_out = torch.nn.Conv2d(
        in_channels=original_conv_out.in_channels,
        out_channels=new_out_channels,
        kernel_size=original_conv_out.kernel_size,
        padding=1,
    )
    new_conv_out.weight.data.zero_()
    new_conv_out.bias.data[: original_conv_out.out_channels] = (
        original_conv_out.bias.data.clone()
    )
    if original_conv_out.out_channels < new_out_channels:
        new_conv_out.weight.data[: original_conv_out.out_channels] = (
            original_conv_out.weight.data
        )
    else:
        new_conv_out.weight.data[:new_out_channels] = original_conv_out.weight.data[
            :new_out_channels
        ]
    return new_conv_out

In [30]:
from leffa.model import SkipAttnProcessor, AttnProcessor2_0

def remove_cross_attention(
    unet,
    cross_attn_cls=SkipAttnProcessor,
    self_attn_cls=None,
    cross_attn_dim=None,
    **kwargs,
):
    if cross_attn_dim is None:
        cross_attn_dim = unet.config.cross_attention_dim
    attn_procs = {}
    for name in unet.attn_processors.keys():
        cross_attention_dim = (
            None if name.endswith("attn1.processor") else cross_attn_dim
        )
        if name.startswith("mid_block"):
            hidden_size = unet.config.block_out_channels[-1]
        elif name.startswith("up_blocks"):
            block_id = int(name[len("up_blocks.")])
            hidden_size = list(reversed(unet.config.block_out_channels))[
                block_id]
        elif name.startswith("down_blocks"):
            block_id = int(name[len("down_blocks.")])
            hidden_size = unet.config.block_out_channels[block_id]
        if cross_attention_dim is None:
            if self_attn_cls is not None:
                attn_procs[name] = self_attn_cls(
                    hidden_size=hidden_size,
                    cross_attention_dim=cross_attention_dim,
                    **kwargs,
                )
            else:
                # retain the original attn processor
                attn_procs[name] = AttnProcessor2_0(
                    hidden_size=hidden_size,
                    cross_attention_dim=cross_attention_dim,
                    layer_name=name,
                    **kwargs,
                )
        else:
            attn_procs[name] = cross_attn_cls(
                hidden_size=hidden_size,
                cross_attention_dim=cross_attention_dim,
                **kwargs,
            )

    unet.set_attn_processor(attn_procs)
    adapter_modules = torch.nn.ModuleList(unet.attn_processors.values())
    return adapter_modules

In [12]:
unet_config, unet_kwargs = ReferenceUNet.load_config(
    pretrained_model_name_or_path,
    subfolder="unet",
    return_unused_kwargs=True,
)
unet_encoder = ReferenceUNet.from_config(
    unet_config, **unet_kwargs)
unet_encoder.config.addition_embed_type = None
unet_encoder.conv_in = replace_conv_in_layer(unet_encoder, 4)
unet_encoder.to('cuda')
remove_cross_attention(unet_encoder, model_type="unet_encoder")

ModuleList(
  (0-31): 32 x None
)

In [13]:
unet_config

{'_class_name': 'UNet2DConditionModel',
 '_diffusers_version': '0.6.0.dev0',
 'act_fn': 'silu',
 'attention_head_dim': 8,
 'block_out_channels': [320, 640, 1280, 1280],
 'center_input_sample': False,
 'cross_attention_dim': 768,
 'down_block_types': ['CrossAttnDownBlock2D',
  'CrossAttnDownBlock2D',
  'CrossAttnDownBlock2D',
  'DownBlock2D'],
 'downsample_padding': 1,
 'flip_sin_to_cos': True,
 'freq_shift': 0,
 'in_channels': 9,
 'layers_per_block': 2,
 'mid_block_scale_factor': 1,
 'norm_eps': 1e-05,
 'norm_num_groups': 32,
 'out_channels': 4,
 'sample_size': 64,
 'up_block_types': ['UpBlock2D',
  'CrossAttnUpBlock2D',
  'CrossAttnUpBlock2D',
  'CrossAttnUpBlock2D']}

In [5]:
import sys
sys.path.append('/workspace/Try-on-Product/projects/Leffa/')
from leffa.model import LeffaModel

vt_model_dc = LeffaModel(
    pretrained_model_name_or_path="../ckpts/stable-diffusion-inpainting",
    pretrained_model="../ckpts/virtual_tryon_dc.pth",
    dtype="float16",
)

In [6]:
pwd

'/workspace/Try-on-Product/projects/Leffa/convert_tensorrt'

In [7]:
input_names = ['sample', 'timestep','encoder_attention_mask']
output_names = ['down', 'reference_features']
# sample_input = (
#     torch.randn(2, 4, 128, 96, dtype=torch.float16, device='cuda'), 
#     torch.tensor([1], dtype=torch.int32, device='cuda'),
#     {
#         'encoder_attention_mask':None
#     }
#     )
sample_input = (
    # torch.randn(2, 4, 128, 96, dtype=torch.float16, device='cuda'), 
    # torch.tensor([1], dtype=torch.int32, device='cuda'),
    {
        'sample':torch.randn(2, 4, 128, 96, dtype=torch.float16, device='cuda'),
        'timestep':torch.tensor([1], dtype=torch.int32, device='cuda'),
        'encoder_attention_mask':None,
    }
    )
onnx_dir = '/workspace/Try-on-Product/projects/Leffa/ckpts/onnx/unet_encoder/'
onnx_opt_dir = '/workspace/Try-on-Product/projects/Leffa/ckpts/onnx/unet_encoder.opt/'
onnx_opset = 18

NameError: name 'torch' is not defined

In [4]:
unet_encoder = vt_model_dc.unet_encoder.cuda()
export_onnx(
    unet_encoder,
    'unet_encoder',
    input_names,
    output_names,
    sample_input,
    onnx_dir,
    onnx_opt_dir,
    onnx_opset,
)

NameError: name 'vt_model_dc' is not defined

### Convert Unet Gen to onnx

In [7]:
unet_gen = vt_model_dc.unet.cuda()

In [8]:
input_names = ['sample', 
               'timestep',
            #    'encoder_hidden_states',
            #    'cross_attention_kwargs',
            #    'added_cond_kwargs',
               'reference_features']
output_names = ['noise_pred']
        # return ((self.xB*batch_size, 3072, 640),)*4 + ((self.xB*batch_size, 768, 1280),)*60 + ((self.xB*batch_size, 3072, 640),)*6,
reference_features_shape= ((2, 12288, 320),)*2 + ((2, 3072, 640),)*2 + ((2, 768, 1280), )*2 + ((2, 192, 1280),) + ((2, 768, 1280), )*3 + ((2, 3072, 640), )*3 + ((2, 12288, 320) ,)*3
sample_input = (
    {
        'sample':torch.randn(2, 12, 128, 96, dtype=torch.float16, device='cuda'),
        'timestep':torch.tensor([1], dtype=torch.int32, device='cuda'),
        'encoder_hidden_states':None,
        'cross_attention_kwargs':None,
        'added_cond_kwargs':None,
        'reference_features':[torch.randn(shape, dtype=torch.float16, device='cuda') for shape in reference_features_shape],
    }
    )
onnx_dir = '/workspace/Try-on-Product/projects/Leffa/ckpts/onnx/unet_gen/'
onnx_opt_dir = '/workspace/Try-on-Product/projects/Leffa/ckpts/onnx/unet_gen.opt/'
onnx_opset = 18

In [10]:
export_onnx(
    unet_gen,
    'unet_gen',
    input_names,
    output_names,
    sample_input,
    onnx_dir,
    onnx_opt_dir,
    onnx_opset,
)

[NeMo W 2025-03-05 15:26:23 nemo_logging:361] /home/thanhdh/.conda/envs/idm-dev/lib/python3.10/site-packages/torch/onnx/utils.py:1548: OnnxExporterWarning: Exporting to ONNX opset version 18 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 17. To use a newer opset version, consider 'torch.onnx.dynamo_export()'. Note that dynamo_export() is in preview. Please report errors with dynamo_export() as Github issues to https://github.com/pytorch/pytorch/issues.
      warnings.warn(
    


[I] Exporting ONNX model: /workspace/Try-on-Product/projects/Leffa/ckpts/onnx/unet_gen/model.onnx
[I] Optimizing ONNX model: /workspace/Try-on-Product/projects/Leffa/ckpts/onnx/unet_gen.opt/model.onnx
unet_gen: original .. 2565 nodes, 3189 tensors, 18 inputs, 1 outputs
unet_gen: cleanup .. 2565 nodes, 3189 tensors, 18 inputs, 1 outputs
[I] Folding Constants | Pass 1
[I]     Total Nodes | Original:  2565, After Folding:  1928 |   637 Nodes Folded
[I] Folding Constants | Pass 2


2025-03-05 15:26:52.758094958 [W:onnxruntime:, constant_folding.cc:269 ApplyImpl] Could not find a CPU kernel and hence can't constant fold Sqrt node '/up_blocks.3/attentions.2/transformer_blocks.0/attn1/Sqrt'
2025-03-05 15:26:52.758357234 [W:onnxruntime:, constant_folding.cc:269 ApplyImpl] Could not find a CPU kernel and hence can't constant fold Sqrt node '/up_blocks.3/attentions.1/transformer_blocks.0/attn1/Sqrt'
2025-03-05 15:26:52.758500511 [W:onnxruntime:, constant_folding.cc:269 ApplyImpl] Could not find a CPU kernel and hence can't constant fold Sqrt node '/up_blocks.3/attentions.0/transformer_blocks.0/attn1/Sqrt'
2025-03-05 15:26:52.758641522 [W:onnxruntime:, constant_folding.cc:269 ApplyImpl] Could not find a CPU kernel and hence can't constant fold Sqrt node '/up_blocks.2/attentions.2/transformer_blocks.0/attn1/Sqrt'
2025-03-05 15:26:52.758788090 [W:onnxruntime:, constant_folding.cc:269 ApplyImpl] Could not find a CPU kernel and hence can't constant fold Sqrt node '/up_block

[I]     Total Nodes | Original:  1928, After Folding:  1627 |   301 Nodes Folded
[I] Folding Constants | Pass 3
[I]     Total Nodes | Original:  1627, After Folding:  1627 |     0 Nodes Folded
unet_gen: fold constants .. 1627 nodes, 2895 tensors, 18 inputs, 1 outputs
unet_gen: shape inference .. 1627 nodes, 2895 tensors, 18 inputs, 1 outputs
unet_gen: finished .. 1627 nodes, 2895 tensors, 18 inputs, 1 outputs


## Export TensorRT Engine

In [11]:
from utilities import Engine
import tensorrt as trt
# Build TensorRT engines
engine_dir = '/workspace/Try-on-Product/projects/Leffa/ckpts/engines/'
do_engine_refit = {
    'unet_encoder': False,
    'unet_gen': False,
}
engines = {}

In [12]:
model_name = "unet_encoder"
optimization_level = 3
fp16= True
bf16= False
def export_engine(model_name):
    engine_name = model_name +'.trt'+trt.__version__+'.plan'
    engine_path = os.path.join(engine_dir,engine_name)
    engine = Engine(engine_path)
    if not os.path.exists(engine_path):
        update_output_names = None
        fp16amp = fp16
        bf16amp = bf16
        strongly_typed = False
        extra_build_args = {'verbose': False}
        extra_build_args['builder_optimization_level'] = optimization_level
        # if use_int8[model_name]:
        #     extra_build_args['int8'] = True
        #     extra_build_args['precision_constraints'] = 'prefer'
        # engine.build(onnx_opt_path[model_name],
        onnx_opt_path = os.path.join(onnx_opt_dir, 'model.onnx')
        engine.build(onnx_opt_path,
            strongly_typed=strongly_typed,
            fp16=fp16amp,
            bf16=bf16amp,
            input_profile=None,
            enable_refit=do_engine_refit[model_name],
            enable_all_tactics=False,
            timing_cache=None,
            update_output_names=update_output_names,
            **extra_build_args)
        engines[model_name]=engine

### Export Unet Encoder Engine

In [18]:
export_engine(model_name)

In [61]:
engine = Engine('/workspace/Try-on-Product/projects/Leffa/ckpts/engines/unet_encoder.trt10.6.0.plan')

In [ ]:
engine.load()

Loading TensorRT engine to cpu bytes: /workspace/Try-on-Product/projects/Leffa/ckpts/engines/unet_encoder.trt10.6.0.plan
[I] Loading bytes from /workspace/Try-on-Product/projects/Leffa/ckpts/engines/unet_encoder.trt10.6.0.plan
Loading TensorRT engine from bytes: /workspace/Try-on-Product/projects/Leffa/ckpts/engines/unet_encoder.trt10.6.0.plan


: 

### Export Unet Generator Engine

In [13]:
export_engine('unet_gen')

Building TensorRT engine for /workspace/Try-on-Product/projects/Leffa/ckpts/onnx/unet_gen.opt/model.onnx: /workspace/Try-on-Product/projects/Leffa/ckpts/engines/unet_gen.trt10.6.0.plan


In [22]:
engine = Engine('/workspace/Try-on-Product/projects/Leffa/ckpts/engines/unet_gen.trt10.6.0.plan')

In [23]:
engine.load()

Loading TensorRT engine to cpu bytes: /workspace/Try-on-Product/projects/Leffa/ckpts/engines/unet_gen.trt10.6.0.plan
[I] Loading bytes from /workspace/Try-on-Product/projects/Leffa/ckpts/engines/unet_gen.trt10.6.0.plan
Loading TensorRT engine from bytes: /workspace/Try-on-Product/projects/Leffa/ckpts/engines/unet_gen.trt10.6.0.plan
